# PlanProof Ablation Study Analysis
Evaluating the contribution of each architectural component to planning compliance validation.

In [ ]:
import json
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

# Try importing visualization deps
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import seaborn as sns
    import numpy as np
except ImportError as e:
    print(f"Install eval deps: pip install -e '.[eval]'\nMissing: {e}")

# PlanProof imports
from planproof.evaluation.results import load_all_results
from planproof.evaluation.metrics import (
    compute_confusion_matrix, compute_recall, compute_precision,
    compute_f2_score, compute_automation_rate, bootstrap_ci, cohens_h,
)

# Style
sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150
RESULTS_DIR = Path("../data/results")
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading

In [ ]:
results = load_all_results(RESULTS_DIR)
if not results:
    print("No results found. Run: python scripts/run_ablation.py first")
else:
    print(f"Loaded {len(results)} experiment results")

# Build a DataFrame for analysis
rows = []
for exp in results:
    for rr in exp.rule_results:
        rows.append({
            "config": exp.config_name,
            "set_id": exp.set_id,
            "rule_id": rr.rule_id,
            "ground_truth": rr.ground_truth_outcome,
            "predicted": rr.predicted_outcome,
            "correct": rr.ground_truth_outcome == rr.predicted_outcome,
        })
df = pd.DataFrame(rows)
print(f"Total rule evaluations: {len(df)}")
df.head()

## 2. Configuration Comparison Summary

In [ ]:
CONFIG_ORDER = [
    "naive_baseline", "strong_baseline",
    "ablation_b", "ablation_c", "ablation_d", "full_system"
]
CONFIG_LABELS = {
    "naive_baseline": "Naive LLM",
    "strong_baseline": "Strong LLM (CoT)",
    "ablation_b": "Ablation B\n(No SNKG)",
    "ablation_c": "Ablation C\n(No Gating)",
    "ablation_d": "Ablation D\n(No Assessability)",
    "full_system": "Full System",
}

summary_rows = []
for config in CONFIG_ORDER:
    config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
    if not config_results:
        continue
    cm = compute_confusion_matrix(config_results)
    recall = compute_recall(cm)
    precision = compute_precision(cm)
    f2 = compute_f2_score(cm)
    auto_rate = compute_automation_rate(config_results)
    summary_rows.append({
        "Configuration": CONFIG_LABELS.get(config, config),
        "Recall": f"{recall:.2%}",
        "Precision": f"{precision:.2%}",
        "F2 Score": f"{f2:.2%}",
        "Automation Rate": f"{auto_rate:.2%}",
        "NOT_ASSESSABLE": cm.get("not_assessable", 0),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

> **Note:** Recall, Precision, and F2 Score are only meaningful when the evaluation set contains **both compliant (no-violation) and non-compliant (violation) plan cases**. If all test cases are compliant, all violation-based metrics will be 0 and the comparison across configurations is not informative. Ensure the ablation dataset includes a representative mix before drawing conclusions.

## 3. Component Contribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

metrics = ["Recall", "Precision", "F2 Score"]
colors = sns.color_palette("husl", len(CONFIG_ORDER))

any_nonzero = False  # track whether all bars are zero (no compliant/non-compliant mix)

for ax, metric in zip(axes, metrics):
    vals = []
    labels = []
    for i, config in enumerate(CONFIG_ORDER):
        config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
        if not config_results:
            continue
        cm = compute_confusion_matrix(config_results)
        if metric == "Recall":
            v = compute_recall(cm)
        elif metric == "Precision":
            v = compute_precision(cm)
        else:
            v = compute_f2_score(cm)
        vals.append(v)
        labels.append(CONFIG_LABELS.get(config, config))
        if v > 0:
            any_nonzero = True

    bars = ax.bar(range(len(vals)), vals, color=colors[:len(vals)])
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1.1)
    ax.set_title(metric, fontweight='bold')
    # Add value labels on bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8)

if not any_nonzero:
    fig.text(0.5, 0.01,
             "All metrics are 0 — results may contain only compliant (no-violation) cases.\n"
             "Run ablation with a mix of compliant and non-compliant plans for meaningful bars.",
             ha='center', va='bottom', fontsize=9, color='red', style='italic')

plt.suptitle("Ablation Study: Component Contribution to Compliance Validation", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ablation_comparison.png", bbox_inches='tight', dpi=300)
plt.show()

## 4. Automation Rate vs Accuracy Trade-off
This plot shows the key innovation: by allowing NOT_ASSESSABLE verdicts, the full system
achieves higher precision on the rules it DOES assess, while honestly flagging gaps.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for i, config in enumerate(CONFIG_ORDER):
    config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
    if not config_results:
        continue
    cm = compute_confusion_matrix(config_results)
    recall = compute_recall(cm)
    auto_rate = compute_automation_rate(config_results)
    label = CONFIG_LABELS.get(config, config)
    ax.scatter(auto_rate, recall, s=200, c=[colors[i]], zorder=5, edgecolors='black', linewidths=1)
    ax.annotate(label, (auto_rate, recall), textcoords="offset points",
                xytext=(10, 10), fontsize=9, ha='left')

ax.set_xlabel("Automation Rate (% rules assessable)", fontsize=12)
ax.set_ylabel("Violation Recall", fontsize=12)
ax.set_title("Automation Rate vs Recall Trade-off\nHigher-right is better", fontsize=13, fontweight='bold')
ax.set_xlim(-0.05, 1.15)
ax.set_ylim(-0.05, 1.15)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "automation_vs_recall.png", bbox_inches='tight', dpi=300)
plt.show()

## 5. Per-Rule Performance Heatmap

In [ ]:
rules = sorted(df['rule_id'].unique())
configs_present = [c for c in CONFIG_ORDER if c in df['config'].values]

accuracy_matrix = []
for config in configs_present:
    row = []
    for rule in rules:
        subset = df[(df['config'] == config) & (df['rule_id'] == rule)]
        if len(subset) == 0:
            row.append(float('nan'))
        else:
            row.append(subset['correct'].mean())
    accuracy_matrix.append(row)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(accuracy_matrix, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

ax.set_xticks(range(len(rules)))
ax.set_xticklabels(rules, fontsize=10)
ax.set_yticks(range(len(configs_present)))
ax.set_yticklabels([CONFIG_LABELS.get(c, c) for c in configs_present], fontsize=9)

# Add text annotations
for i in range(len(configs_present)):
    for j in range(len(rules)):
        val = accuracy_matrix[i][j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.0%}', ha='center', va='center', fontsize=10,
                    color='white' if val < 0.5 else 'black')

plt.colorbar(im, label='Accuracy')
ax.set_title("Per-Rule Accuracy by Configuration", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / "per_rule_heatmap.png", bbox_inches='tight', dpi=300)
plt.show()

## 6. NOT_ASSESSABLE Verdict Analysis
The assessability engine is the core research contribution. This section analyses when and why
rules are classified as NOT_ASSESSABLE.

In [ ]:
not_assessable = df[df['predicted'] == 'NOT_ASSESSABLE']
if len(not_assessable) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # By config
    na_by_config = not_assessable.groupby('config').size()
    na_by_config.plot(kind='barh', ax=axes[0], color=sns.color_palette("husl", len(na_by_config)))
    axes[0].set_xlabel("Count of NOT_ASSESSABLE verdicts")
    axes[0].set_title("NOT_ASSESSABLE by Configuration")

    # By rule
    na_by_rule = not_assessable.groupby('rule_id').size()
    na_by_rule.plot(kind='barh', ax=axes[1], color=sns.color_palette("Set2", len(na_by_rule)))
    axes[1].set_xlabel("Count of NOT_ASSESSABLE verdicts")
    axes[1].set_title("NOT_ASSESSABLE by Rule")

    plt.suptitle("Assessability Analysis", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "not_assessable_analysis.png", bbox_inches='tight', dpi=300)
    plt.show()
else:
    print("No NOT_ASSESSABLE verdicts found in results.")

## 7. Statistical Confidence
Bootstrap confidence intervals for key metrics (1000 resamples).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
y_positions = []
y_labels = []

for i, config in enumerate(CONFIG_ORDER):
    config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
    if not config_results:
        continue

    # Bootstrap recall
    recalls = []
    rng = np.random.RandomState(42)
    for _ in range(1000):
        sample = rng.choice(config_results, size=len(config_results), replace=True)
        cm = compute_confusion_matrix(list(sample))
        recalls.append(compute_recall(cm))

    mean = np.mean(recalls)

    # Handle degenerate cases: all recalls identical (no variance) or mean is 0.0
    if mean == 0.0 or len(set(recalls)) == 1:
        lower, upper = mean, mean
    else:
        lower = np.percentile(recalls, 2.5)
        upper = np.percentile(recalls, 97.5)

    # Clamp to avoid negative xerr values (can occur with floating-point edge cases)
    xerr_lo = max(0.0, mean - lower)
    xerr_hi = max(0.0, upper - mean)

    ax.errorbar(mean, i, xerr=[[xerr_lo], [xerr_hi]], fmt='o', markersize=8,
                capsize=5, color=colors[i], linewidth=2)
    y_positions.append(i)
    y_labels.append(CONFIG_LABELS.get(config, config))

ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels)
ax.set_xlabel("Violation Recall (95% Bootstrap CI)")
ax.set_title("Recall with Confidence Intervals", fontsize=13, fontweight='bold')
ax.set_xlim(-0.05, 1.15)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.3, label='50% baseline')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "bootstrap_ci.png", bbox_inches='tight', dpi=300)
plt.show()

## 8. Statistical Comparisons
Cohen's h effect sizes: Full System vs each ablation.

In [ ]:
full_results = [r for exp in results for r in exp.rule_results if exp.config_name == "full_system"]
if full_results:
    full_cm = compute_confusion_matrix(full_results)
    full_recall = compute_recall(full_cm)

    effect_sizes = []
    for config in ["naive_baseline", "strong_baseline", "ablation_b", "ablation_c", "ablation_d"]:
        config_results = [r for exp in results for r in exp.rule_results if exp.config_name == config]
        if not config_results:
            continue
        cm = compute_confusion_matrix(config_results)
        r = compute_recall(cm)
        h = cohens_h(full_recall, r)
        effect_sizes.append({
            "Comparison": f"Full System vs {CONFIG_LABELS.get(config, config)}",
            "Full Recall": f"{full_recall:.2%}",
            "Other Recall": f"{r:.2%}",
            "Cohen's h": f"{h:.3f}",
            "Effect": "Large" if abs(h) > 0.8 else "Medium" if abs(h) > 0.5 else "Small" if abs(h) > 0.2 else "Negligible",
        })
    pd.DataFrame(effect_sizes)

## 9. Qualitative Error Analysis
For each misclassification, document what went wrong and why.

In [ ]:
misclassified = df[~df['correct']]
print(f"Total misclassifications: {len(misclassified)}")
if len(misclassified) > 0:
    for _, row in misclassified.iterrows():
        print(f"\n--- {row['config']} | {row['set_id']} | {row['rule_id']} ---")
        print(f"Ground truth: {row['ground_truth']}, Predicted: {row['predicted']}")
        print("Analysis: [TODO: explain why this happened]")

## 10. Key Findings for Dissertation

### Summary
[Fill in after running experiments]

### Tables for LaTeX export

In [ ]:
# Export summary table for LaTeX
if len(summary_df) > 0:
    print(summary_df.to_latex(index=False, caption="Ablation Study Results", label="tab:ablation"))
    summary_df.to_csv(FIGURES_DIR / "summary_table.csv", index=False)
    print(f"\nFigures saved to {FIGURES_DIR}")

In [ ]:
# ── Figure S7: Concordance Heatmap ───────────────────────────────────────────
# Rows: rules (sorted), Cols: SABLE-enabled configs
# Cell value: mean SABLE belief across all test sets for that (rule, config) pair.
# Diverging colormap centred at the mid-point of the observed range.
# Configs with no belief data (ablation_a) shown as hatched NaN cells.

_hm_configs = [c for c in SABLE_CONFIGS if c in df_sable['config'].unique()]
_rules       = sorted(df_sable['rule_id'].unique())

# Build matrix: shape (n_rules, n_configs)
_matrix = np.full((len(_rules), len(_hm_configs)), np.nan)
for ci, cfg in enumerate(_hm_configs):
    _sub_cfg = df_sable[df_sable['config'] == cfg]
    for ri, rule in enumerate(_rules):
        vals = _sub_cfg[(_sub_cfg['rule_id'] == rule) & _sub_cfg['belief'].notna()]['belief']
        if len(vals) > 0:
            _matrix[ri, ci] = vals.mean()

# Colormap: RdYlGn diverging, centred at 0.5
_cmap = plt.cm.RdYlGn
_vmin, _vmax = 0.0, 1.0

fig, ax = plt.subplots(figsize=(max(6, len(_hm_configs) * 2 + 1), max(4, len(_rules) * 0.7 + 1.5)))
im = ax.imshow(_matrix, cmap=_cmap, vmin=_vmin, vmax=_vmax, aspect='auto')

# Hatch NaN cells
for ri in range(len(_rules)):
    for ci in range(len(_hm_configs)):
        if np.isnan(_matrix[ri, ci]):
            ax.add_patch(plt.Rectangle(
                (ci - 0.5, ri - 0.5), 1, 1,
                fill=True, facecolor='#EEEEEE', edgecolor='#BDBDBD',
                linewidth=0.5, hatch='///',
            ))
            ax.text(ci, ri, 'N/A', ha='center', va='center',
                    fontsize=8, color='#888', style='italic')
        else:
            # Annotate with value
            _bg = _matrix[ri, ci]
            _text_color = 'white' if (_bg < 0.3 or _bg > 0.75) else 'black'
            ax.text(ci, ri, f'{_bg:.2f}', ha='center', va='center',
                    fontsize=8.5, color=_text_color, fontweight='bold')

ax.set_xticks(range(len(_hm_configs)))
ax.set_xticklabels([CONFIG_LABELS.get(c, c) for c in _hm_configs], fontsize=10)
ax.set_yticks(range(len(_rules)))
ax.set_yticklabels(_rules, fontsize=10)

ax.set_xlabel('Configuration')
ax.set_ylabel('Rule ID')
ax.set_title('Mean SABLE Belief Score per Rule × Configuration\n'
             'Diverging colormap: red = low belief (blocked), green = high belief (confident)')

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('Mean Bel(sufficient)', fontsize=9)

plt.tight_layout()
_out = SABLE_FIGURES / 'sable_concordance_heatmap.png'
plt.savefig(_out)
print(f"Saved → {_out}")
plt.show()

### Figure S7 — Concordance Heatmap (Mean Belief per Rule × Config)

In [ ]:
# ── Figure S6: Component Contribution Table ──────────────────────────────────
# Rendered as a matplotlib figure for dissertation inclusion.

# Build results_by_config dict required by compute_component_contribution
from planproof.evaluation.results import RuleResult as _RR

_results_by_config: dict[str, list] = {}
for exp in all_results:
    _results_by_config.setdefault(exp.config_name, []).extend(exp.rule_results)

_contrib_rows = compute_component_contribution(_results_by_config, baseline_config='full_system')

# Column display names
_COL_KEYS    = ['component_removed', 'recall_delta', 'precision_delta',
                'f2_delta', 'false_fail_delta', 'not_assessable_delta',
                'mcnemar_p', 'cohens_h']
_COL_HEADERS = ['Component Removed', 'Recall Δ', 'Precision Δ',
                'F2 Δ', 'False FAILs Δ', 'N/A Δ',
                'McNemar p', "Cohen's h"]

def _fmt_cell(key, val):
    if val is None:
        return '—'
    if isinstance(val, float):
        import math
        if math.isnan(val):
            return 'n/a'
        if key in ('recall_delta', 'precision_delta', 'f2_delta'):
            sign = '+' if val >= 0 else ''
            return f'{sign}{val:.3f}'
        if key == 'mcnemar_p':
            return f'{val:.4f}' if val >= 0.0001 else '<0.0001'
        return f'{val:+.3f}' if key != 'component_removed' else str(val)
    return str(val)

_table_data  = [[_fmt_cell(k, row.get(k)) for k in _COL_KEYS] for row in _contrib_rows]
_n_rows      = len(_table_data)
_n_cols      = len(_COL_HEADERS)

fig, ax = plt.subplots(figsize=(14, max(3, _n_rows * 0.9 + 1.5)))
ax.axis('off')

tbl = ax.table(
    cellText=_table_data,
    colLabels=_COL_HEADERS,
    cellLoc='center',
    loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.6)

# Style header row
for col_idx in range(_n_cols):
    cell = tbl[0, col_idx]
    cell.set_facecolor('#1565C0')
    cell.set_text_props(color='white', fontweight='bold')

# Color-code data cells: green for improvement, red for degradation, neutral otherwise
_IMPROVEMENT_COLS = {1, 2, 3}   # Recall Δ, Precision Δ, F2 Δ → positive = good
_DEGRADATION_COLS = {4, 5}      # False FAILs Δ, N/A Δ → positive = bad

for row_idx, row in enumerate(_contrib_rows, start=1):
    for col_idx, key in enumerate(_COL_KEYS):
        val = row.get(key)
        cell = tbl[row_idx, col_idx]
        cell.set_facecolor('#F5F5F5' if row_idx % 2 == 0 else 'white')
        if isinstance(val, float) and not (isinstance(val, float) and
                                           __import__('math').isnan(val)):
            if col_idx in _IMPROVEMENT_COLS:
                if val > 0.01:
                    cell.set_facecolor('#C8E6C9')   # light green
                elif val < -0.01:
                    cell.set_facecolor('#FFCDD2')   # light red
            elif col_idx in _DEGRADATION_COLS:
                if val > 0:
                    cell.set_facecolor('#FFCDD2')
                elif val < 0:
                    cell.set_facecolor('#C8E6C9')

ax.set_title('Component Contribution: Full System vs Each Ablation\n'
             'Δ = Full System minus Ablation (positive = component improves metric)',
             fontsize=11, fontweight='bold', pad=20)

plt.tight_layout()
_out = SABLE_FIGURES / 'sable_component_contribution.png'
plt.savefig(_out)
print(f"Saved → {_out}")

# Also print as DataFrame for notebook readability
import pandas as pd
print(pd.DataFrame(_contrib_rows).to_string(index=False))
plt.show()

### Figure S6 — Component Contribution Table

In [ ]:
# ── Figure S5: False-FAIL Prevention Bar Chart ───────────────────────────────
# False FAILs = cases where ground_truth=PASS but predicted=FAIL (FP in CM).

_ff_configs = [c for c in ALL_CONFIGS if c in df_sable['config'].unique()]
_ff_counts  = []
for cfg in _ff_configs:
    _sub = df_sable[df_sable['config'] == cfg]
    n_fp = (((_sub['ground_truth'] == 'PASS') & (_sub['predicted'] == 'FAIL'))).sum()
    _ff_counts.append(int(n_fp))

fig, ax = plt.subplots(figsize=(9, 5))
_x = np.arange(len(_ff_configs))
_bar_colors = [CONFIG_COLORS.get(c, '#888') for c in _ff_configs]

bars = ax.bar(_x, _ff_counts, color=_bar_colors, edgecolor='white',
              linewidth=0.8, alpha=0.88)

# Highlight full_system bar with a bold border
_fs_idx = _ff_configs.index('full_system') if 'full_system' in _ff_configs else -1
if _fs_idx >= 0:
    bars[_fs_idx].set_edgecolor('#1565C0')
    bars[_fs_idx].set_linewidth(2.5)

# Value labels on bars
for bar, val in zip(bars, _ff_counts):
    ax.text(bar.get_x() + bar.get_width() / 2.,
            bar.get_height() + max(_ff_counts) * 0.015,
            str(val), ha='center', va='bottom', fontsize=11, fontweight='bold')

# Zero annotation for full_system
if _fs_idx >= 0 and _ff_counts[_fs_idx] == 0:
    ax.annotate(
        'Zero false FAILs\n(SABLE prevents over-flagging)',
        xy=(_fs_idx, 0), xytext=(_fs_idx + 0.5, max(_ff_counts) * 0.4),
        arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5),
        fontsize=9, color='#1565C0',
    )

ax.set_xticks(_x)
ax.set_xticklabels([CONFIG_LABELS.get(c, c) for c in _ff_configs], fontsize=10)
ax.set_xlabel('Configuration')
ax.set_ylabel('False FAIL Count (FP: gt=PASS, pred=FAIL)')
ax.set_title('False-FAIL Prevention by Configuration\n'
             'SABLE assessability engine eliminates spurious violation flags')
ax.set_ylim(0, max(_ff_counts) * 1.25 if max(_ff_counts) > 0 else 5)

plt.tight_layout()
_out = SABLE_FIGURES / 'sable_false_fail_prevention.png'
plt.savefig(_out)
print(f"Saved → {_out}")
plt.show()

### Figure S5 — False-FAIL Prevention

In [ ]:
# ── Figure S4: Blocking Reason Stacked Bar ───────────────────────────────────
# Only SABLE-enabled configs carry blocking_reason values.
# Possible values: NONE, LOW_CONFIDENCE, CONFLICTING_EVIDENCE, MISSING_EVIDENCE,
# plus null (configs where SABLE is disabled).

_BLOCKING_ORDER  = ['NONE', 'LOW_CONFIDENCE', 'CONFLICTING_EVIDENCE', 'MISSING_EVIDENCE']
_BLOCKING_COLORS = {
    'NONE':                 '#43A047',
    'LOW_CONFIDENCE':       '#FFA726',
    'CONFLICTING_EVIDENCE': '#EF5350',
    'MISSING_EVIDENCE':     '#AB47BC',
}

_br_configs = [c for c in SABLE_CONFIGS if c in df_sable['config'].unique()]

# Build count matrix
_br_data = {}
for cfg in _br_configs:
    _sub = df_sable[df_sable['config'] == cfg]
    _dist = blocking_reason_distribution(
        [type('R', (), {'blocking_reason': v})() for v in _sub['blocking_reason'].tolist()]
    )
    _br_data[cfg] = {k: _dist.get(k, 0) for k in _BLOCKING_ORDER}
    # also collect 'null' separately for annotation
    _br_data[cfg]['null'] = _dist.get('null', 0)

fig, ax = plt.subplots(figsize=(10, 6))
_x = np.arange(len(_br_configs))
_bottom = np.zeros(len(_br_configs))

for reason in _BLOCKING_ORDER:
    vals = np.array([_br_data[c].get(reason, 0) for c in _br_configs], dtype=float)
    color = _BLOCKING_COLORS.get(reason, '#888')
    ax.bar(_x, vals, bottom=_bottom, color=color, alpha=0.88,
           edgecolor='white', linewidth=0.8, label=reason.replace('_', ' ').title())
    for xi, (v, b) in enumerate(zip(vals, _bottom)):
        if v > 0:
            ax.text(xi, b + v / 2, str(int(v)), ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    _bottom = _bottom + vals

# Annotate null count (configs with no blocking_reason field) below x-axis
for xi, cfg in enumerate(_br_configs):
    n_null = _br_data[cfg].get('null', 0)
    if n_null > 0:
        ax.text(xi, -_bottom.max() * 0.04, f'(+{n_null} null)',
                ha='center', va='top', fontsize=7, color='grey')

ax.set_xticks(_x)
ax.set_xticklabels([CONFIG_LABELS.get(c, c) for c in _br_configs], fontsize=10)
ax.set_xlabel('Configuration')
ax.set_ylabel('Number of Rule Evaluations')
ax.set_title('SABLE Blocking Reason Distribution\n'
             'Explains why evidence sufficiency was not reached')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, _bottom.max() * 1.14)

plt.tight_layout()
_out = SABLE_FIGURES / 'sable_blocking_reasons.png'
plt.savefig(_out)
print(f"Saved → {_out}")
plt.show()

### Figure S4 — Blocking Reason Stacked Bar

In [ ]:
# ── Figure S3: Belief vs Plausibility Scatter ────────────────────────────────
# Points on the diagonal (Bel = Pl) indicate zero ignorance mass.
# Points above the diagonal are impossible (Bel ≤ Pl always).
# The vertical gap Pl - Bel = ignorance (uncommitted mass).

_df_scatter = df_sable[
    df_sable['belief'].notna() & df_sable['plausibility'].notna()
].copy()

_scatter_configs = [c for c in SABLE_CONFIGS if c in _df_scatter['config'].unique()]

fig, ax = plt.subplots(figsize=(7, 6))

for cfg in _scatter_configs:
    _sub = _df_scatter[_df_scatter['config'] == cfg]
    # Jitter slightly to reveal overlapping points
    _rng = np.random.RandomState(hash(cfg) & 0xFFFF)
    _jitter = _rng.uniform(-0.008, 0.008, len(_sub))
    ax.scatter(
        _sub['belief'].values + _jitter,
        _sub['plausibility'].values + _jitter,
        s=30, alpha=0.65,
        color=CONFIG_COLORS.get(cfg, '#888'),
        edgecolors='none',
        label=CONFIG_LABELS.get(cfg, cfg).replace('\n', ' '),
        zorder=3,
    )

# Diagonal: Bel == Pl (zero ignorance)
ax.plot([0, 1], [0, 1], color='#555', linestyle='--', linewidth=1.1,
        label='Zero ignorance (Bel = Pl)', zorder=2)

# Shade the valid region Bel ≤ Pl
ax.fill_between([0, 1], [0, 0], [0, 1], alpha=0.04, color='steelblue')

ax.set_xlabel('Bel(sufficient) — Belief')
ax.set_ylabel('Pl(sufficient) — Plausibility')
ax.set_xlim(-0.05, 1.08)
ax.set_ylim(-0.05, 1.08)
ax.set_title('SABLE Belief vs Plausibility\n'
             'Gap above diagonal = ignorance mass (uncommitted evidence)')
ax.legend(fontsize=9, loc='upper left')
ax.set_aspect('equal')

plt.tight_layout()
_out = SABLE_FIGURES / 'sable_belief_vs_plausibility.png'
plt.savefig(_out)
print(f"Saved → {_out}")
plt.show()

### Figure S3 — Belief vs Plausibility Scatter

In [ ]:
# ── Figure S2: Three-State Stacked Bar Chart ─────────────────────────────────
# Verdicts collapse to: ASSESSABLE (PASS+FAIL), PARTIALLY_ASSESSABLE, NOT_ASSESSABLE

_configs_present = [c for c in ALL_CONFIGS if c in df_sable['config'].unique()]

def _classify_verdict(pred):
    if pred in ('PASS', 'FAIL'):
        return 'ASSESSABLE'
    return pred  # PARTIALLY_ASSESSABLE or NOT_ASSESSABLE

_df_v = df_sable[df_sable['config'].isin(_configs_present)].copy()
_df_v['verdict_class'] = _df_v['predicted'].map(_classify_verdict)

_counts = (
    _df_v.groupby(['config', 'verdict_class'])
         .size()
         .unstack(fill_value=0)
         .reindex(_configs_present)
)

# Ensure all three columns exist
for _col in ['ASSESSABLE', 'PARTIALLY_ASSESSABLE', 'NOT_ASSESSABLE']:
    if _col not in _counts.columns:
        _counts[_col] = 0
_counts = _counts[['ASSESSABLE', 'PARTIALLY_ASSESSABLE', 'NOT_ASSESSABLE']]

_state_colors  = ['#43A047', '#FFA726', '#EF5350']  # green / amber / red
_state_labels  = ['Assessable (PASS + FAIL)', 'Partially Assessable', 'Not Assessable']

fig, ax = plt.subplots(figsize=(10, 6))
_x = np.arange(len(_configs_present))
_bottom = np.zeros(len(_configs_present))

for col, color, label in zip(_counts.columns, _state_colors, _state_labels):
    vals = _counts[col].values
    ax.bar(_x, vals, bottom=_bottom, color=color, alpha=0.88,
           edgecolor='white', linewidth=0.8, label=label)
    # Value labels inside each segment
    for xi, (v, b) in enumerate(zip(vals, _bottom)):
        if v > 0:
            ax.text(xi, b + v / 2, str(int(v)), ha='center', va='center',
                    fontsize=9, color='white', fontweight='bold')
    _bottom = _bottom + vals

ax.set_xticks(_x)
ax.set_xticklabels([CONFIG_LABELS.get(c, c) for c in _configs_present], fontsize=10)
ax.set_xlabel('Configuration')
ax.set_ylabel('Number of Rule Evaluations')
ax.set_title('Verdict Distribution Across Configurations\n'
             '(ASSESSABLE = binary PASS/FAIL; PARTIALLY_ASSESSABLE = low-confidence; '
             'NOT_ASSESSABLE = insufficient evidence)')
ax.legend(loc='upper right', fontsize=9)
ax.set_ylim(0, _bottom.max() * 1.12)

plt.tight_layout()
_out = SABLE_FIGURES / 'sable_three_state_bar.png'
plt.savefig(_out)
print(f"Saved → {_out}")
plt.show()

### Figure S2 — Three-State Verdict Stacked Bar Chart

In [ ]:
# ── Figure S1: Belief Distribution Violin Plot ───────────────────────────────
# Only SABLE-enabled configs have non-null belief scores.
# Ablation A (no VLM) produces null beliefs → excluded (plotted as annotation).
# Falls back from violin → box → strip if fewer than 2 unique values per group.

_belief_configs = [c for c in SABLE_CONFIGS
                   if df_sable[df_sable['config'] == c]['belief'].notna().any()]

_df_belief = df_sable[df_sable['config'].isin(_belief_configs) &
                      df_sable['belief'].notna()].copy()

fig, ax = plt.subplots(figsize=(9, 5))

_positions = list(range(len(_belief_configs)))
_plot_type  = 'none'

if HAS_SEABORN and len(_df_belief) >= 4:
    try:
        sns.violinplot(
            data=_df_belief, x='config', y='belief',
            order=_belief_configs,
            palette={c: CONFIG_COLORS.get(c, '#888') for c in _belief_configs},
            inner='box', cut=0, linewidth=1.2, ax=ax,
        )
        _plot_type = 'violin'
    except Exception:
        pass

if _plot_type == 'none':
    # Fallback: box plot
    _grouped = [_df_belief[_df_belief['config'] == c]['belief'].values
                for c in _belief_configs]
    _has_var = [len(np.unique(g)) > 1 for g in _grouped]
    if all(_has_var):
        bp = ax.boxplot(_grouped, positions=_positions, patch_artist=True, widths=0.5)
        for patch, cfg in zip(bp['boxes'], _belief_configs):
            patch.set_facecolor(CONFIG_COLORS.get(cfg, '#888'))
            patch.set_alpha(0.7)
        _plot_type = 'box'
    else:
        # Last resort: strip / scatter
        for pos, cfg in zip(_positions, _belief_configs):
            vals = _df_belief[_df_belief['config'] == cfg]['belief'].values
            jitter = np.random.RandomState(0).uniform(-0.15, 0.15, len(vals))
            ax.scatter(pos + jitter, vals, s=18, alpha=0.6,
                       color=CONFIG_COLORS.get(cfg, '#888'), edgecolors='none')
        _plot_type = 'strip'

# Threshold reference lines
ax.axhline(0.7, color='#43A047', linestyle='--', linewidth=1.2,
           label='Assessable threshold (0.7)')
ax.axhline(0.3, color='#E53935', linestyle='--', linewidth=1.2,
           label='Not-assessable threshold (0.3)')

ax.set_xticks(_positions)
ax.set_xticklabels([CONFIG_LABELS.get(c, c) for c in _belief_configs], fontsize=10)
ax.set_xlabel('Configuration')
ax.set_ylabel('Bel(sufficient) — SABLE belief score')
ax.set_ylim(-0.05, 1.10)
ax.set_title(f'Distribution of SABLE Belief Scores Across Configurations\n'
             f'({_plot_type} plot; configs with null beliefs omitted)')
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
_out = SABLE_FIGURES / 'sable_belief_violin.png'
plt.savefig(_out)
print(f"Saved → {_out}")
plt.show()

### Figure S1 — Belief Distribution (Violin / Box / Strip)

In [ ]:
# ── SABLE setup: dissertation styling + data loading ────────────────────────
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

from planproof.evaluation.results import load_all_results
from planproof.evaluation.metrics import (
    compute_confusion_matrix, compute_recall, compute_precision,
    compute_f2_score, blocking_reason_distribution, compute_component_contribution,
    mcnemar_test, cohens_h,
)

# Dissertation-quality global style
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

CONFIG_COLORS = {
    'full_system':  '#2196F3',
    'ablation_a':   '#FF9800',
    'ablation_b':   '#4CAF50',
    'ablation_c':   '#9C27B0',
    'ablation_d':   '#F44336',
    'naive_baseline':   '#795548',
    'strong_baseline':  '#607D8B',
}
SABLE_CONFIGS = ['full_system', 'ablation_a', 'ablation_b', 'ablation_c']
ALL_CONFIGS   = ['full_system', 'ablation_a', 'ablation_b', 'ablation_c', 'ablation_d']

CONFIG_LABELS = {
    'full_system':  'Full System',
    'ablation_a':   'Ablation A\n(No VLM)',
    'ablation_b':   'Ablation B\n(No SNKG)',
    'ablation_c':   'Ablation C\n(No Gating)',
    'ablation_d':   'Ablation D\n(No Assessability)',
}

# Output directory — relative to the notebook (../figures from notebooks/)
SABLE_FIGURES = Path('../figures')
SABLE_FIGURES.mkdir(exist_ok=True)

# Load results
_results_dir = Path('../data/results')
all_results = load_all_results(_results_dir)
print(f"Loaded {len(all_results)} experiment results")

# Build flat DataFrame with SABLE fields
sable_rows = []
for exp in all_results:
    for rr in exp.rule_results:
        sable_rows.append({
            'config':          rr.config_name,
            'rule_id':         rr.rule_id,
            'set_id':          rr.set_id,
            'belief':          rr.belief,
            'plausibility':    rr.plausibility,
            'conflict_mass':   rr.conflict_mass,
            'blocking_reason': rr.blocking_reason,
            'predicted':       rr.predicted_outcome,
            'ground_truth':    rr.ground_truth_outcome,
        })

df_sable = pd.DataFrame(sable_rows)
print(f"Total rule evaluations: {len(df_sable)}")
print(df_sable.groupby('config')['predicted'].value_counts().to_string())

## SABLE Evidence Sufficiency Analysis

The following seven figures examine the Dempster-Shafer belief-plausibility metrics produced by the SABLE assessability engine. Each figure is dissertation-ready (300 DPI, serif typeface).